In [ ]:
!pip install unsloth

In [ ]:
import unsloth
import transformers

In [ ]:
from pathlib import Path

CONFIG = {
    # --- paths ---
    "data_dir":    Path("/kaggle/input/datasets/eazdanmostafarafin/bangla-meme-classification-dataset/preprocessed"),
    "output_dir":  Path("/kaggle/working/outputs"),

    # --- resume from checkpoint ---
    # Set to the checkpoint folder to resume from (e.g. checkpoint-578 = end of epoch 2).
    # Set to None for a fresh run from scratch.
    # On Kaggle, upload the checkpoint as a dataset and point here, e.g.
    #   Path("/kaggle/input/vlm-checkpoint-578/checkpoint-578")

    # uncomment this line if checkpoint is available
    # "resume_from_checkpoint": Path("/kaggle/input/datasets/eazdanmostafarafin/bangla-meme-vlm-ft-checkpoints/checkpoint-578"),

    # --- model ---
    "model_name":     "unsloth/Qwen2-VL-2B-Instruct",
    "load_in_4bit":   True,
    "max_seq_length": 1024,

    # --- split ---
    "test_size":        0.2,
    "val_frac_of_temp": 0.5,
    "seed":             42,

    # --- QLoRA ---
    "lora_r":       16,
    "lora_alpha":   32,
    "lora_dropout": 0.05,

    # --- training (QLoRA-tuned) ---
    # NOTE: num_train_epochs is the TOTAL across all runs, not additional.
    # When resuming from checkpoint-578 (end of epoch 2), setting this to 7
    # means the trainer runs epochs 3 -> 7 (5 more epochs).
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size":  1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs":  2,
    "learning_rate":     2e-4,
    "warmup_ratio":      0.03,
    "max_grad_norm":     0.3,
    "weight_decay":      0.01,
    "lr_scheduler_type": "cosine",
    "optim":             "paged_adamw_8bit",
    "logging_steps":     5,
    "eval_strategy":     "epoch",
    "save_strategy":     "steps",
    "save_steps":        100,
    "save_total_limit":  3,
}

CONFIG["image_dir"] = CONFIG["data_dir"] / "Image_processed"
CONFIG["csv_path"]  = CONFIG["data_dir"] / "Train_processed.csv"
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = CONFIG["output_dir"]


In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    CONFIG["model_name"],
    load_in_4bit = CONFIG["load_in_4bit"],
)


In [ ]:
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

IMAGE_DIR = CONFIG["image_dir"]
CSV_PATH  = CONFIG["csv_path"]

df = pd.read_csv(CSV_PATH)
df = df[["Image_name", "Target"]].dropna()
df = df[df["Image_name"].apply(lambda n: (IMAGE_DIR / n).is_file())].reset_index(drop=True)

LABELS = sorted(df["Target"].unique().tolist())
print(f"Classes: {LABELS}")
print(f"Total usable samples: {len(df)}")

train_df, temp_df = train_test_split(
    df, test_size=CONFIG["test_size"], stratify=df["Target"], random_state=CONFIG["seed"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=CONFIG["val_frac_of_temp"], stratify=temp_df["Target"], random_state=CONFIG["seed"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


In [ ]:


INSTRUCTION = (
    "You are an image classifier. Look at the image and classify it into exactly "
    f"one of these categories: {', '.join(LABELS)}. "
    "Respond with only the single category name."
)

def row_to_conversation(row):
    image = Image.open(IMAGE_DIR / row["Image_name"]).convert("RGB")
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": INSTRUCTION},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": str(row["Target"])},
                ],
            },
        ]
    }

train_dataset = [row_to_conversation(r) for _, r in train_df.iterrows()]
val_dataset   = [row_to_conversation(r) for _, r in val_df.iterrows()]
print(f"Train convos: {len(train_dataset)} | Val convos: {len(val_dataset)}")



In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = CONFIG["lora_r"],
    lora_alpha     = CONFIG["lora_alpha"],
    lora_dropout   = CONFIG["lora_dropout"],
    bias           = "none",
    random_state   = CONFIG["seed"],
    use_rslora     = False,
    loftq_config   = None,
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    args = SFTConfig(
        per_device_train_batch_size = CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size  = CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"],
        warmup_ratio                = CONFIG["warmup_ratio"],
        num_train_epochs            = CONFIG["num_train_epochs"],
        learning_rate               = CONFIG["learning_rate"],
        max_grad_norm               = CONFIG["max_grad_norm"],
        fp16                        = not is_bf16_supported(),
        bf16                        = is_bf16_supported(),
        logging_steps               = CONFIG["logging_steps"],
        logging_strategy            = "steps",
        disable_tqdm                = True,
        optim                       = CONFIG["optim"],
        weight_decay                = CONFIG["weight_decay"],
        lr_scheduler_type           = CONFIG["lr_scheduler_type"],
        seed                        = CONFIG["seed"],
        output_dir                  = str(CONFIG["output_dir"]),
        logging_dir                 = str(CONFIG["output_dir"] / "logs"),
        report_to                   = "none",
        remove_unused_columns       = False,
        dataset_text_field          = "",
        dataset_kwargs              = {"skip_prepare_dataset": True},
        dataset_num_proc            = 1,
        max_seq_length              = CONFIG["max_seq_length"],
        eval_strategy               = CONFIG["eval_strategy"],
        save_strategy               = CONFIG["save_strategy"],
        save_steps                  = CONFIG["save_steps"],
        save_total_limit            = CONFIG["save_total_limit"],
    ),
)


In [ ]:
import json

OUTPUT_DIR = CONFIG["output_dir"]
adapter_dir = OUTPUT_DIR / "final_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

# Resolve resume path: must exist on disk, else fall back to fresh run.
resume_ckpt = CONFIG.get("resume_from_checkpoint")
if resume_ckpt is not None:
    resume_ckpt = Path(resume_ckpt)
    if not resume_ckpt.is_dir():
        print(f"[resume] Checkpoint not found at {resume_ckpt}, starting fresh.")
        resume_ckpt = None
    else:
        print(f"[resume] Resuming training from: {resume_ckpt}")

train_metrics = {}
eval_metrics = {}
training_completed = False

try:
    trainer_stats = trainer.train(
        resume_from_checkpoint=str(resume_ckpt) if resume_ckpt else None,
    )
    print(trainer_stats)
    train_metrics = trainer_stats.metrics
    trainer.log_metrics("train", train_metrics)
    trainer.save_metrics("train", train_metrics)
    training_completed = True
except KeyboardInterrupt:
    print("Training interrupted. Saving current model state...")
finally:
    trainer.save_state()
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print(f"Saved LoRA adapter + tokenizer to: {adapter_dir.resolve()}")

# Pull the last eval_* entry from the log_history that was recorded
# during training (eval_strategy="epoch"). Avoids re-calling
# trainer.evaluate(), which errors with "on_train_begin must be called
# before on_evaluate" when invoked outside the training loop.
for entry in reversed(trainer.state.log_history):
    if any(k.startswith("eval_") for k in entry):
        eval_metrics = {k: v for k, v in entry.items() if k.startswith("eval_")}
        break

if eval_metrics:
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)

if not training_completed:
    interrupt_info = {
        "status": "interrupted",
        "global_step": int(trainer.state.global_step),
        "epoch": float(trainer.state.epoch) if trainer.state.epoch is not None else None,
    }
    with (OUTPUT_DIR / "interrupt_summary.json").open("w", encoding="utf-8") as f:
        json.dump(interrupt_info, f, indent=2)
    print(f"Saved interruption summary to: {(OUTPUT_DIR / 'interrupt_summary.json').resolve()}")

summary = {
    "train": train_metrics,
    "eval": eval_metrics,
    "training_completed": training_completed,
}
with (OUTPUT_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved trainer artifacts under: {OUTPUT_DIR.resolve()}")

try:
    merged_dir = OUTPUT_DIR / "final_merged_16bit"
    model.save_pretrained_merged(
        str(merged_dir),
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"Saved merged full model to: {merged_dir.resolve()}")
except Exception as e:
    print("Merged full-model export was skipped:", e)


In [ ]:
import json
import pandas as pd
import torch
from tqdm import tqdm

OUTPUT_DIR  = CONFIG["output_dir"]
IMAGE_DIR   = CONFIG["image_dir"]
RESULTS_DIR = OUTPUT_DIR / "eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FastVisionModel.for_inference(model)

def predict_label(image, instruction):
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction},
        ]},
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors     = "pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 16,
            do_sample      = False,
            temperature    = 0.0,
            use_cache      = True,
        )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

def normalize_to_label(pred_text, labels):
    low = pred_text.lower()
    for lab in labels:
        if lab.lower() in low:
            return lab
    return pred_text

correct = 0
total   = 0
per_class_correct = {lab: 0 for lab in LABELS}
per_class_total   = {lab: 0 for lab in LABELS}
prediction_rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    image = Image.open(IMAGE_DIR / row["Image_name"]).convert("RGB")
    raw_pred = predict_label(image, INSTRUCTION)
    pred     = normalize_to_label(raw_pred, LABELS)
    true     = row["Target"]
    is_correct = int(pred == true)

    per_class_total[true] += 1
    total += 1
    if is_correct:
        correct += 1
        per_class_correct[true] += 1

    prediction_rows.append({
        "Image_name": row["Image_name"],
        "Target": true,
        "Prediction": pred,
        "Raw_Prediction": raw_pred,
        "Correct": is_correct,
    })

overall_acc = correct / total if total else 0.0
per_class_metrics = {}
for lab in LABELS:
    n = per_class_total[lab]
    acc = per_class_correct[lab] / n if n else 0.0
    per_class_metrics[lab] = {
        "correct": per_class_correct[lab],
        "total": n,
        "accuracy": acc,
    }

predictions_path = RESULTS_DIR / "test_predictions.csv"
metrics_path = RESULTS_DIR / "test_metrics.json"

pd.DataFrame(prediction_rows).to_csv(predictions_path, index=False)
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "overall": {
                "correct": correct,
                "total": total,
                "accuracy": overall_acc,
            },
            "per_class": per_class_metrics,
        },
        f,
        indent=2,
    )

print(f"\nTest accuracy: {correct}/{total} = {overall_acc:.4f}")
for lab in LABELS:
    m = per_class_metrics[lab]
    print(f"  {lab:10s}: {m['correct']}/{m['total']} = {m['accuracy']:.4f}")

print(f"Saved test predictions to: {predictions_path.resolve()}")
print(f"Saved test metrics to: {metrics_path.resolve()}")
